In [ ]:
"""

Title: "Scalable Neural Safety Certification via Monotonicity"
Code for: Learing a neural barrier certificate for Lotka–Volterra (2000D) example.
Author: Amirreza Alavi
Contact: seyedamirreza.alavi@colorado.edu
Affiliation: University of Colorado Boulder
Date: 04/2026

"""

In [ ]:
import tensorflow as tf 
from tensorflow import keras
from tensorflow.keras import layers, constraints, initializers
import matplotlib.pyplot as plt
from numpy import linalg as LA
from scipy import stats
from functools import partial
%matplotlib inline
import numpy as np
from scipy.optimize import fsolve
from tensorflow.keras.layers import Input, Dense, Concatenate, Add
from tensorflow.keras import Model
from tensorflow.keras import regularizers
from tensorflow.keras.models import load_model
import time
from mpl_toolkits import mplot3d
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from mpl_toolkits.mplot3d import Axes3D  
from math import inf
import heapq
from dataclasses import dataclass
import matplotlib.patches as mpatches

In [ ]:
# Define useful functions for generating a Metzler A and discritization parameter tau for monotone dtSYS 

t1 = time.perf_counter()

def sample_A_r(N, rng,
               offdiag_range=(0.0001, 0.0005),  
               diag_range=(5, 6),      
               r_range=(1, 2)):      
    A = rng.uniform(offdiag_range[0], offdiag_range[1], size=(N, N))

    diag = rng.uniform(diag_range[0], diag_range[1], size=N)
    np.fill_diagonal(A, -diag)

    r = rng.uniform(r_range[0], r_range[1], size=N)
    return A, r

def is_hurwitz(A, tol=1e-12):
    lam = np.linalg.eigvals(A)
    return np.all(np.real(lam) < -tol)

def check_Ar(A, r, tol=1e-12):
    x_star = -np.linalg.solve(A, r)      
    return np.all(x_star > tol), x_star

def tau_max_for_monotonicity(A, r, xbar):

    A = np.asarray(A, dtype=float)
    r = np.asarray(r, dtype=float).reshape(-1)
    aii = np.diag(A)

    g = r + 2.0 * aii * xbar  
    neg = g < 0
    if not np.any(neg):
        return np.inf

    return np.min(1.0 / (-g[neg]))

def check_monotone_on_box(A, r, xbar, tau, num_tests=20000, seed=0):

    rng = np.random.default_rng(seed)
    N = len(r)
    for _ in range(num_tests):
        x = rng.uniform(0.0, xbar, size=N)
        Ax_r = A @ x + r
        DF = np.eye(N) + tau * (np.diag(x) @ A + np.diag(Ax_r))

        if np.min(DF) < -1e-10:  
            return False
    return True

In [ ]:
# Checking A to be Hurwitz and having a positive eq point

N = 2000
A, r = sample_A_r(N, rng=np.random.default_rng(7))

print("Hurwitz:", is_hurwitz(A))
okA2, xstar = check_Ar(A, r)
print("(-A^{-1}r > 0):", okA2)
# print("x* =", xstar)

xbar = 1.0
tau_max = tau_max_for_monotonicity(A, r, xbar)
print("tau_max for monotonicity", tau_max)
tau = np.floor(0.5 * tau_max * 1e2) / 1e2 
print("tau", tau)

In [ ]:
# System Definition and inductive evaluation functions

def f_lv(x):

    A, r = sample_A_r(N, rng=np.random.default_rng(7))
    xbar = 1.0
    tau_max = tau_max_for_monotonicity(A, r, xbar)
    tau = np.floor(0.5 * tau_max * 1e2) / 1e2 
    
    x = tf.convert_to_tensor(x, dtype=tf.float32)
    A = tf.convert_to_tensor(A, dtype=tf.float32)
    r = tf.convert_to_tensor(r, dtype=tf.float32)
    tau = tf.cast(tau, tf.float32)

    squeeze = (x.shape.rank == 1)
    if squeeze:
        x = x[None, :]  

    Ax = tf.linalg.matmul(x, A, transpose_b=True) 
    Ax_r = Ax + r[None, :]                        

    x_next = x + tau * (x * Ax_r)                 

    return x_next[0] if squeeze else x_next

def f_lv_0km1(x, k):
    x = tf.convert_to_tensor(x, dtype=tf.float32)
    if x.shape.rank == 1:
        x = tf.expand_dims(q, 0)     
    B, N = x.shape
    traj = tf.TensorArray(tf.float32, size=k, element_shape=(B, N))
    for k in tf.range(k):
        traj = traj.write(k, x)      
        x = f_lv(x)                  
    return traj.stack() 


def f_lv_k(x, k):
    q = tf.convert_to_tensor(x, dtype=tf.float32)
    for _ in tf.range(k):
        q = f_lv(q)
    return q

In [ ]:
# Sampled points for a coarse gridding

xin_bar = np.ones((1,N))*0.4
xin_und = np.zeros((1,N))
xuns_und = np.eye(N)*0.5
xuns_bar = np.ones((1,N))*1
xi_bar= np.ones((1,N))*0.5
xi_und= np.zeros((1,N))

k=100
f_lv_0km1_xin_bar = tf.squeeze(f_lv_0km1(xin_bar,k))
f_lv_0km1_xi_und = tf.squeeze(f_lv_0km1(xi_und,k), axis=1)
f_lv_k_xi_bar = f_lv_k(xi_bar,k)

In [ ]:
# Neural Barrier Definition

class ScalarMonotoneNN(layers.Layer):

    def __init__(self, hidden_dims=(200, 200), activation=None, output_activation='linear'):
        super().__init__()

        if activation is None:
            activation = tf.keras.layers.LeakyReLU(alpha=0.1)

        self.hidden_layers = []
        for h in hidden_dims:
            self.hidden_layers.append(
                layers.Dense(
                    h,
                    activation=activation,
                    use_bias=True,
                    kernel_constraint=constraints.NonNeg()
                )
            )

        self.out_layer = layers.Dense(
            1,
            activation=output_activation,
            use_bias=True,
            kernel_constraint=constraints.NonNeg()
        )

    def call(self, x):
        z = x
        for layer in self.hidden_layers:
            z = layer(z)
        return self.out_layer(z)


class MaxAffineBranch(layers.Layer):

    def __init__(self, m_i, hidden_dims=(200, 200), activation=None, output_activation='linear'):
        super().__init__()

        self.affine = layers.Dense(
            m_i,
            activation=None,
            use_bias=True,
            kernel_constraint=constraints.NonNeg()
        )

        self.scalar_nn = ScalarMonotoneNN(
            hidden_dims=hidden_dims,
            activation=activation,
            output_activation=output_activation
        )

    def call(self, x):
        z = self.affine(x)                            
        g = tf.reduce_max(z, axis=1, keepdims=True)  
        y = self.scalar_nn(g)                        
        return y


class InputMaxBranch(layers.Layer):

    def __init__(self, hidden_dims=(200, 200), activation=None, output_activation='linear'):
        super().__init__()

        self.scalar_nn = ScalarMonotoneNN(
            hidden_dims=hidden_dims,
            activation=activation,
            output_activation=output_activation
        )

    def call(self, x):
        g = tf.reduce_max(x, axis=1, keepdims=True)  
        y = self.scalar_nn(g)                        
        return y


class ThreeBranchMinMonotoneNetwork(Model):

    def __init__(self,
                 input_dim,
                 m1,
                 m2,
                 hidden_dims1=(200, 200),
                 hidden_dims2=(200, 200),
                 hidden_dims3=(200, 200),
                 activation=None,
                 output_activation='linear'):
        super().__init__()

        self.input_dim = input_dim


        self.branch1 = InputMaxBranch(
            hidden_dims=hidden_dims1,
            activation=activation,
            output_activation=output_activation
        )


        self.branch2 = MaxAffineBranch(
            m_i=m1,
            hidden_dims=hidden_dims2,
            activation=activation,
            output_activation=output_activation
        )


        self.branch3 = MaxAffineBranch(
            m_i=m2,
            hidden_dims=hidden_dims3,
            activation=activation,
            output_activation=output_activation
        )

    def call(self, x):
        y1 = self.branch1(x)   
        y2 = self.branch2(x)   
        y3 = self.branch3(x)   

        Y = tf.concat([y1, y2, y3], axis=1)          
        y = tf.reduce_min(Y, axis=1, keepdims=True)  
        return y


modelB = ThreeBranchMinMonotoneNetwork(
    input_dim=N,
    m1=int(0.1*N),   
    m2=int(0.2*N),   
    hidden_dims1=(200, 200),     
    hidden_dims2=(200, 200),
    hidden_dims3=(200, 200),
    activation=tf.keras.layers.LeakyReLU(alpha=0.1),
    output_activation='linear'
)

In [ ]:
# Training Monotone Neural Inductive Barrier

epochs = 100000
opt1 = tf.keras.optimizers.Adam(1e-2)

gamma = -10
eta = 10


for i in range(epochs):
    with tf.GradientTape() as tape0, tf.GradientTape() as tape1:

        loss_B = tf.reduce_sum(tf.nn.relu(tf.subtract(1.0,1.0)))

        loss_B += tf.square(tf.nn.relu(tf.reduce_max(modelB(f_lv_0km1_xin_bar)) - gamma))
        loss_B += tf.square(tf.nn.relu(eta - tf.reduce_min(modelB(xuns_und))))
        loss_B += tf.math.sign(tf.nn.relu(-(tf.reduce_max(modelB(f_lv_0km1_xi_und)) - 0.0)))  * (tf.nn.relu(modelB(f_lv_k_xi_bar) -  0.0))

                                
        grad_B = tape0.gradient(loss_B, modelB.trainable_variables)
                
        if loss_B == 0   and i>1:
            print('Safety Verified')
            break
            
        if loss_B!=0:
            opt1.apply_gradients(zip(grad_B, modelB.trainable_variables),)
            
        if i%5 == 1:
            print(
            f"Epoch {i}: "
            f"maxB[xinitil_bar(0 to k-1)] = {tf.reduce_max(modelB(f_lv_0km1_xin_bar)).numpy():.8f}, "
            f"minB[xunsafe_und] = {tf.reduce_min(modelB(xuns_und)).numpy():.8f}, "
            f"Third condition = {tf.math.sign(tf.nn.relu(-(tf.reduce_max(modelB(f_lv_0km1_xi_und)) - 0.0)))  * (tf.nn.relu(modelB(f_lv_k_xi_bar) -  0.0)).numpy():}, "
            f"LossB = {loss_B}"
            )


    
t2 = time.perf_counter()
tot = t2 - t1
print(f'Total runtime: {tot}s')
print(
            f"Epoch {i}: "
            f"maxB[xinitil_bar(0 to k-1)] = {tf.reduce_max(modelB(f_lv_0km1_xin_bar)).numpy():.8f}, "
            f"minB[xunsafe_und] = {tf.reduce_min(modelB(xuns_und)).numpy():.8f}, "
            f"Third condition = {tf.math.sign(tf.nn.relu(-(tf.reduce_max(modelB(f_lv_0km1_xi_und)) - 0.0)))  * (tf.nn.relu(modelB(f_lv_k_xi_bar) -  0.0)).numpy():}, "
            f"LossB = {loss_B}"
            )